# Notebook 01 — RQ1 & RQ4: File Granularity and Scalability

Covers:
- **RQ1**: How file count / size affects DuckDB query performance (flat A-series models)
- **RQ4**: Whether performance inflection points shift between SF-1 and SF-10
- **RQ5**: Memory (peak RSS) overhead as a function of file count

Produces:
- Table 1 (paper): cold-cache median wall-clock matrix, SF-10
- Figure: wall-clock latency vs model for all 8 queries (SF-10)
- Figure: Q1 scalability ratio SF-10 / SF-1
- Figure: peak RAM vs model (SF-10)
- Wilcoxon significance tests for all key pairwise comparisons


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import wilcoxon
from itertools import combinations

CSV_PATH = '../results/benchmark_results.csv'
FIGURES_DIR = '../figures'
import os; os.makedirs(FIGURES_DIR, exist_ok=True)

df   = pd.read_csv(CSV_PATH)
cold = df[df['RunType'] == 'COLD'].copy()
warm = df[df['RunType'] == 'WARM'].copy()

FLAT_MODELS  = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7']
FLAT_QUERIES = ['Q1', 'Q2a', 'Q2b', 'Q2c', 'Q4', 'Q5', 'Q6', 'Q7']

MODEL_LABELS = {
    'A1': 'A1\n1×7.5 GB',
    'A2': 'A2\n8×950 MB',
    'A3': 'A3\n38×200 MB',
    'A4': 'A4\n75×100 MB',
    'A5': 'A5\n150×50 MB',
    'A6': 'A6\n750×10 MB',
    'A7': 'A7\n7500×1 MB',
}

print('Data loaded:', len(cold), 'cold-cache rows')

## 1  Table 1 — Cold-cache median wall-clock (ms), SF-10, all flat models

In [ ]:
sf10_flat = cold[
    (cold['SF'] == 10) &
    (cold['Model'].isin(FLAT_MODELS)) &
    (cold['Query'].isin(FLAT_QUERIES))
]

table1 = (
    sf10_flat
    .groupby(['Model', 'Query'])['WallClock_ms']
    .median()
    .unstack()[FLAT_QUERIES]
    .reindex(FLAT_MODELS)
    .round(0)
    .astype(int)
)

print('Table 1 — Median wall-clock time (ms), cold cache, SF-10')
print(table1.to_string())
table1.to_csv(f'{FIGURES_DIR}/table1_rq1_sf10_medians.csv')

## 2  Standard deviations table

In [ ]:
table1_std = (
    sf10_flat
    .groupby(['Model', 'Query'])['WallClock_ms']
    .std()
    .unstack()[FLAT_QUERIES]
    .reindex(FLAT_MODELS)
    .round(1)
)
print('Standard deviations (ms):')
print(table1_std.to_string())
table1_std.to_csv(f'{FIGURES_DIR}/table1_rq1_sf10_stds.csv')

## 3  Figure — Wall-clock latency vs model, all queries, SF-10 (log scale)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=False)
axes = axes.flatten()

for idx, query in enumerate(FLAT_QUERIES):
    ax = axes[idx]
    sub = sf10_flat[sf10_flat['Query'] == query]
    medians = sub.groupby('Model')['WallClock_ms'].median().reindex(FLAT_MODELS)
    stds    = sub.groupby('Model')['WallClock_ms'].std().reindex(FLAT_MODELS)

    x = range(len(FLAT_MODELS))
    ax.bar(x, medians.values, yerr=stds.values, capsize=4,
           color='#2E75B6', alpha=0.85, error_kw={'elinewidth': 1})
    ax.set_yscale('log')
    ax.set_title(query, fontweight='bold')
    ax.set_xticks(list(x))
    ax.set_xticklabels([m.replace('A','A') for m in FLAT_MODELS], fontsize=8)
    ax.set_ylabel('Wall-clock (ms)' if idx % 4 == 0 else '')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda v, _: f'{v:,.0f}' if v >= 1000 else f'{v:.0f}'
    ))
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    # Mark the A4→A5 cliff
    ax.axvline(x=3.5, color='red', linestyle=':', linewidth=1.2, alpha=0.7)

fig.suptitle(
    'Cold-cache median wall-clock time by model — SF-10 (log scale)\n'
    'Red dashed line marks the A4→A5 performance cliff',
    fontsize=12
)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_rq1_wallclock_sf10.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/fig_rq1_wallclock_sf10.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved fig_rq1_wallclock_sf10')

## 4  Figure — Q4 (metadata-only) across all models, both SFs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)

for ax, sf in zip(axes, [1, 10]):
    models_sf = [m for m in FLAT_MODELS if m != 'A2'] if sf == 1 else FLAT_MODELS
    sub = cold[(cold['SF'] == sf) & (cold['Query'] == 'Q4') &
               (cold['Model'].isin(models_sf))]
    medians = sub.groupby('Model')['WallClock_ms'].median().reindex(models_sf)
    stds    = sub.groupby('Model')['WallClock_ms'].std().reindex(models_sf)

    ax.bar(range(len(models_sf)), medians.values, yerr=stds.values,
           capsize=4, color='#C55A11', alpha=0.85)
    ax.set_yscale('log')
    ax.set_title(f'Q4 (COUNT*) — SF-{sf}', fontweight='bold')
    ax.set_xticks(range(len(models_sf)))
    ax.set_xticklabels(models_sf)
    ax.set_ylabel('Wall-clock (ms)')
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Metadata overhead (Q4) — direct proxy for per-file parsing cost', fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_rq1_q4_metadata.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/fig_rq1_q4_metadata.png', dpi=200, bbox_inches='tight')
plt.show()

## 5  Figure — Peak RSS memory, SF-10, Q1

In [ ]:
mem_q1 = cold[(cold['SF'] == 10) & (cold['Query'] == 'Q1') &
               (cold['Model'].isin(FLAT_MODELS))]
mem_q7 = cold[(cold['SF'] == 10) & (cold['Query'] == 'Q7') &
               (cold['Model'].isin(FLAT_MODELS))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, qdata, qtitle in zip(axes,
    [mem_q1, mem_q7],
    ['Q1 (full-scan aggregation)', 'Q7 (join)']):

    medians_gb = qdata.groupby('Model')['PeakRAM_KB'].median().reindex(FLAT_MODELS) / (1024**2)
    stds_gb    = qdata.groupby('Model')['PeakRAM_KB'].std().reindex(FLAT_MODELS)    / (1024**2)

    ax.bar(range(len(FLAT_MODELS)), medians_gb.values, yerr=stds_gb.values,
           capsize=4, color='#70AD47', alpha=0.85)
    ax.set_yscale('log')
    ax.set_title(f'Peak RSS — {qtitle}', fontweight='bold')
    ax.set_xticks(range(len(FLAT_MODELS)))
    ax.set_xticklabels(FLAT_MODELS)
    ax.set_ylabel('Peak RAM (GB)')
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Peak RSS memory by model — SF-10, cold cache (log scale)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_rq5_memory_sf10.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/fig_rq5_memory_sf10.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nMedian peak RAM (GB) — Q1 SF-10:')
print(mem_q1.groupby('Model')['PeakRAM_KB'].median().reindex(FLAT_MODELS).div(1024**2).round(2))

## 6  Figure — Scalability: SF-10 / SF-1 ratio per model (Q1)

In [ ]:
SCALABILITY_MODELS = ['A1', 'A3', 'A4', 'A5', 'A6']  # A2 only at SF-10; A7 included below

ratios = {}
for model in SCALABILITY_MODELS + ['A7']:
    sf1_rows = cold[(cold['SF']==1)  & (cold['Model']==model) & (cold['Query']=='Q1')]
    sf10_rows= cold[(cold['SF']==10) & (cold['Model']==model) & (cold['Query']=='Q1')]
    if sf1_rows.empty:
        continue
    r = sf10_rows['WallClock_ms'].median() / sf1_rows['WallClock_ms'].median()
    ratios[model] = r

models_plot = list(ratios.keys())
ratio_vals  = [ratios[m] for m in models_plot]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(range(len(models_plot)), ratio_vals, color='#2E75B6', alpha=0.85)
ax.axhline(y=10, color='red', linestyle='--', linewidth=1.5, label='Expected 10× (linear scaling)')
ax.set_xticks(range(len(models_plot)))
ax.set_xticklabels(models_plot)
ax.set_ylabel('SF-10 / SF-1 latency ratio (Q1)')
ax.set_title('Scalability ratio — Q1 (full-scan)\nRed line = expected linear scaling', fontweight='bold')
ax.set_yscale('log')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)

for bar, val in zip(bars, ratio_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f'{val:.1f}×', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_rq4_scalability.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/fig_rq4_scalability.png', dpi=200, bbox_inches='tight')
plt.show()

print('Scalability table (Q1 cold medians):')
for m, r in ratios.items():
    sf1  = cold[(cold['SF']==1) &(cold['Model']==m)&(cold['Query']=='Q1')]['WallClock_ms'].median()
    sf10 = cold[(cold['SF']==10)&(cold['Model']==m)&(cold['Query']=='Q1')]['WallClock_ms'].median()
    print(f'  {m}: SF-1={sf1:.1f} ms   SF-10={sf10:.1f} ms   ratio={r:.1f}×')

## 7  Warm-cache improvement analysis

In [ ]:
print('Warm vs cold cache improvement — Q1, SF-10:')
print(f'{"Model":<6} {"Cold median (ms)":>18} {"Warm median (ms)":>18} {"Improvement %":>15}')
print('-' * 62)
for model in FLAT_MODELS:
    c = cold[(cold['SF']==10)&(cold['Model']==model)&(cold['Query']=='Q1')]['WallClock_ms'].median()
    w = warm[(warm['SF']==10)&(warm['Model']==model)&(warm['Query']=='Q1')]['WallClock_ms'].median()
    pct = (c - w) / c * 100
    print(f'{model:<6} {c:>18.1f} {w:>18.1f} {pct:>14.0f}%')

## 8  Wilcoxon signed-rank tests — RQ1 key comparisons

In [ ]:
def wilcoxon_pair(label_a, label_b, model_a, query_a, model_b, query_b, sf=10):
    a = cold[(cold['SF']==sf)&(cold['Model']==model_a)&(cold['Query']==query_a)]['WallClock_ms'].values
    b = cold[(cold['SF']==sf)&(cold['Model']==model_b)&(cold['Query']==query_b)]['WallClock_ms'].values
    if len(a) != len(b):
        print(f'  SKIP {label_a} vs {label_b}: unequal sample sizes ({len(a)} vs {len(b)})')
        return
    stat, p = wilcoxon(a, b)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    med_a, med_b = np.median(a), np.median(b)
    print(f'  {label_a:<25} vs {label_b:<25}  '
          f'med={med_a:.0f} vs {med_b:.0f} ms   '
          f'W={stat:.0f}  p={p:.6f}  {sig}')

print('=== RQ1: Adjacent model comparisons (Q1, SF-10) ===')
pairs = list(zip(FLAT_MODELS, FLAT_MODELS[1:]))
for m1, m2 in pairs:
    wilcoxon_pair(f'{m1}/Q1', f'{m2}/Q1', m1, 'Q1', m2, 'Q1')

print()
print('=== RQ1: A4 vs A5 for all queries (SF-10) ===')
for q in FLAT_QUERIES:
    wilcoxon_pair(f'A4/{q}', f'A5/{q}', 'A4', q, 'A5', q)

print()
print('=== RQ1: Selectivity comparison within A1 (SF-10) ===')
for q1, q2 in [('Q2a','Q2b'), ('Q2a','Q2c'), ('Q2b','Q2c')]:
    wilcoxon_pair(f'A1/{q1}', f'A1/{q2}', 'A1', q1, 'A1', q2)